# Query a REST API in Python with chDB (pandas API)

Companion to [query-rest-api-with-sql-python](https://clickhouse.com/resources/engineering/query-rest-api-with-sql-python).

Run `./generate.sh` first to create `data/`. Requires `pip install chdb pandas`.

This notebook starts a local `http.server` over `data/` so `DataStore.from_url()` has a real HTTP endpoint to hit. The same call works unchanged against any JSON API.

In [ ]:
import http.server
import threading
import time

from chdb.datastore import DataStore

PORT = 8731
BASE = f"http://127.0.0.1:{PORT}"

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *a, **k):
        super().__init__(*a, directory="data", **k)
    def log_message(self, *a, **k):
        pass
    def handle_one_request(self):
        try:
            super().handle_one_request()
        except (BrokenPipeError, ConnectionResetError):
            self.close_connection = True

httpd = http.server.ThreadingHTTPServer(("127.0.0.1", PORT), QuietHandler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
time.sleep(0.3)
print(f"Serving data/ on {BASE}")

## 1. Read a JSON API response into a DataFrame

In [ ]:
df = DataStore.from_url(f"{BASE}/orders.json", format="JSONEachRow")
df

In [ ]:
df.dtypes

## 2. Filter + aggregate the way you already do (pandas, not SQL)

In [ ]:
revenue = (
    df[df["status"] != "closed"]
    .groupby("country")["amount"]
    .sum()
    .sort_values(ascending=False)
)
revenue.to_pandas()

## 3. Hand off to real pandas when you need it

In [ ]:
pdf = df.to_pandas()   # a real pandas.DataFrame, in memory
type(pdf)

## 4. Multiple pages: read each URL, then concat

In [ ]:
import pandas as real_pd

page1 = DataStore.from_url(f"{BASE}/orders_page1.json", format="JSONEachRow").to_pandas()
page2 = DataStore.from_url(f"{BASE}/orders_page2.json", format="JSONEachRow").to_pandas()
pages = real_pd.concat([page1, page2], ignore_index=True)
pages.groupby("country")["amount"].sum().sort_values(ascending=False)

## 5. Performance: DataStore.from_url vs urllib + json + manual aggregation

Apple M4 Pro (14 cores, 24 GB RAM, macOS); chDB 4.1.8, Python 3.14; best-of-3 warm, over localhost: ~12-13x faster than fetch-then-parse-then-loop.

In [ ]:
import json
import urllib.request

def datastore_agg():
    d = DataStore.from_url(f"{BASE}/orders_large.json", format="JSONEachRow")
    r = (
        d[d["status"] == "open"]
        .groupby("country")["amount"]
        .sum()
        .sort_values(ascending=False)
    )
    return r.to_pandas()

def requests_agg():
    agg = {}
    with urllib.request.urlopen(f"{BASE}/orders_large.json") as resp:
        for line in resp:
            r = json.loads(line)
            if r["status"] == "open":
                a = agg.setdefault(r["country"], [0, 0.0])
                a[0] += 1
                a[1] += r["amount"]
    return agg

def best_of_3(fn):
    fn()
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

ds_s = best_of_3(datastore_agg)
py_s = best_of_3(requests_agg)
print(f"requests + json + manual agg:   {py_s:.3f}s")
print(f"DataStore.from_url (chDB):      {ds_s:.3f}s")
print(f"speedup:                        {py_s / ds_s:.1f}x")